# Premier League 2026/27 — hourly idempotent ingest

**What this notebook does**

1. Downloads the latest Premier League CSV from [football-data.co.uk](https://www.football-data.co.uk/englandm.php) (free for personal / educational use — no scraping, one HTTPS `GET` per run).
2. Parses it into competitions / teams / matches rows.
3. Walks the season chronologically updating Elo ratings per match.
4. Writes the full retrospective probability timeline (`pre-match` + 5-minute reconstructed grid + `half-time` + `full-time` + a snapshot at each goal / red-card minute; scorelines come from the ESPN `match_events` when they agree with the CSV final score, else the HT/FT approximation) using the Poisson-race predictor shared with the Go backend.
5. Builds a market-consensus benchmark row per match from the Avg / Bet365 pre-match odds columns.
6. Upserts everything idempotently via `INSERT ... ON DUPLICATE KEY UPDATE` + deterministic SHA-1-derived UUIDs. Safe to re-run every hour.

**Scheduling** (in SingleStore Portal)

1. Save this notebook into a workspace that has the `singlescore` database attached.
2. Open **Jobs** → **Create Job** → pick this notebook, set a 1-hour cron (`0 * * * *`), and save.
3. The job inherits the workspace's connection; the notebook picks it up automatically via `SINGLESTOREDB_URL`.

**Local dev**

The same body lives at `scripts/ingest_pl_season.py` so you can run it from a laptop with:

```
python3 scripts/ingest_pl_season.py
```

It reads `.env` for `SINGLESTORE_HOST` / `SINGLESTORE_USER` / `SINGLESTORE_PASSWORD` when run outside the Portal.

## 1 – Dependencies

SingleStore Portal has `pandas` and `requests` pre-installed; `pymysql` is small and installs in a second if it's missing.

In [ ]:
import importlib, subprocess, sys
for pkg in ("pandas", "requests", "pymysql"):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", pkg])
print("deps ok")

## 2 – Config + helpers

The constants here (`MODEL_NAME`, `MODEL_VERSION`, `AVG_GOALS_PER_TEAM`, `ELO_HALF_LIFE`, `RED_CARD_FACTOR`, `HOME_ADVANTAGE`) **must** stay in sync with `backend/internal/prediction/baseline.go` so a row written by the notebook is indistinguishable from one written by the Go backend.

In [ ]:
import csv, hashlib, io, json, math, os, sys, time, uuid
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Iterable

import pandas as pd
import pymysql
import requests

CSV_URL = os.environ.get(
    "PL_CSV_URL", "https://www.football-data.co.uk/mmz4281/2627/E0.csv"
)
PROVIDER = "football-data-couk"
COMPETITION_EXT_ID = "E0-2026-27"
COMPETITION_NAME = "Premier League"
COMPETITION_COUNTRY = "England"
COMPETITION_SEASON = "2026/27"

MODEL_NAME = "elo-poisson"
MODEL_VERSION = "v0.2.0"
AVG_GOALS_PER_TEAM = 1.35
ELO_HALF_LIFE = 800.0
RED_CARD_FACTOR = 0.72
MAX_EXTRA_GOALS = 10
HOME_ADVANTAGE = 60.0
ELO_K = 20.0
INITIAL_ELO = 1500.0
TIMELINE_STEP_MIN = 5

def det_id(prefix: str, key: str) -> str:
    h = hashlib.sha1(f"{prefix}|{key}".encode("utf-8")).digest()[:16]
    return str(uuid.UUID(bytes=h))

def connect():
    url = os.environ.get("SINGLESTOREDB_URL") or os.environ.get("SINGLESTORE_URL")
    if url:
        from urllib.parse import urlparse
        p = urlparse(url)
        return pymysql.connect(
            host=p.hostname, port=p.port or 3306,
            user=p.username, password=p.password or "",
            database=(p.path or "/").lstrip("/") or "singlescore",
            autocommit=True, ssl={"ssl": True},
            charset="utf8mb4", use_unicode=True,
        )
    # Local fallback: read env (loaded from .env below for script runs).
    return pymysql.connect(
        host=os.environ["SINGLESTORE_HOST"],
        port=int(os.environ.get("SINGLESTORE_PORT", "3306")),
        user=os.environ["SINGLESTORE_USER"],
        password=os.environ["SINGLESTORE_PASSWORD"],
        database=os.environ.get("SINGLESTORE_DATABASE", "singlescore"),
        autocommit=True, ssl={"ssl": True},
        charset="utf8mb4", use_unicode=True,
    )

print("config ok, model=", MODEL_NAME, MODEL_VERSION)

## 3 – Poisson-race predictor (Python port of the Go baseline)

`λ_home = 1.35 · exp(+gap/2)` and `λ_away = 1.35 · exp(−gap/2)` where `gap = (home_elo + home_adv − away_elo) / 800`.  λ is scaled by `(90−minute)/90` so residual Poisson variance is 0 at minute 90 → the leader commits to 1.0 at FT with no special-casing.

In [ ]:
_FACT = [1, 1, 2, 6, 24, 120, 720, 5040, 40320, 362880, 3628800]

def _poisson_pmf_vec(mu: float):
    out = [0.0] * (MAX_EXTRA_GOALS + 1)
    if mu <= 0:
        out[0] = 1.0; return out
    e_neg = math.exp(-mu); pow_mu = 1.0
    for k in range(MAX_EXTRA_GOALS + 1):
        out[k] = e_neg * pow_mu / _FACT[k]; pow_mu *= mu
    return out

def predict(home_elo, away_elo, home_score, away_score, minute, home_red=0, away_red=0):
    gap = (home_elo + HOME_ADVANTAGE - away_elo) / ELO_HALF_LIFE
    lh = AVG_GOALS_PER_TEAM * math.exp(+gap / 2)
    la = AVG_GOALS_PER_TEAM * math.exp(-gap / 2)
    m = max(0, min(90, minute))
    remaining = (90 - m) / 90.0
    imb = home_red - away_red
    if imb > 0: lh *= RED_CARD_FACTOR ** imb
    elif imb < 0: la *= RED_CARD_FACTOR ** (-imb)
    mu_h = lh * remaining; mu_a = la * remaining
    current = home_score - away_score
    p_h = _poisson_pmf_vec(mu_h); p_a = _poisson_pmf_vec(mu_a)
    p_home = p_draw = p_away = 0.0
    for h in range(MAX_EXTRA_GOALS + 1):
        for a in range(MAX_EXTRA_GOALS + 1):
            p = p_h[h] * p_a[a]
            margin = current + (h - a)
            if margin > 0: p_home += p
            elif margin < 0: p_away += p
            else: p_draw += p
    s = p_home + p_draw + p_away
    if s <= 0 or math.isnan(s): return (1/3, 1/3, 1/3)
    return (p_home / s, p_draw / s, p_away / s)

def _elo_update(home_elo, away_elo, home_goals, away_goals):
    expected_home = 1.0 / (1.0 + 10 ** ((away_elo - home_elo - HOME_ADVANTAGE) / 400.0))
    if home_goals > away_goals: r = 1.0
    elif home_goals < away_goals: r = 0.0
    else: r = 0.5
    gd = abs(home_goals - away_goals)
    k = ELO_K * (1 + math.log(gd + 1))
    delta = k * (r - expected_home)
    return home_elo + delta, away_elo - delta

## 4 – Fetch + parse CSV

In [ ]:
print(f"GET {CSV_URL}")
r = requests.get(CSV_URL, timeout=30, allow_redirects=True); r.raise_for_status()
df = pd.read_csv(io.StringIO(r.text))
df["Date"] = pd.to_datetime(df["Date"], format="%d/%m/%Y", errors="coerce")
df = df.dropna(subset=["Date", "HomeTeam", "AwayTeam"]).copy()
if "Time" not in df.columns: df["Time"] = "15:00"
df["Time"] = df["Time"].fillna("15:00")
df["KickoffAt"] = pd.to_datetime(
    df["Date"].dt.strftime("%Y-%m-%d") + " " + df["Time"].astype(str), errors="coerce"
)
df["KickoffAt"] = df["KickoffAt"].dt.tz_localize(None)
df = df.dropna(subset=["KickoffAt"]).sort_values("KickoffAt").reset_index(drop=True)
played_mask = ~df["FTHG"].isna()
print(f"rows parsed: {len(df)}  played: {int(played_mask.sum())}  unplayed: {int((~played_mask).sum())}")
df.head()

## 5 – Build all rows (teams, matches, event-driven timelines, benchmarks)

In [ ]:
comp_id = det_id("competition", f"{PROVIDER}|{COMPETITION_EXT_ID}")
now_utc = datetime.now(timezone.utc).replace(tzinfo=None)
team_names = sorted(set(df["HomeTeam"]) | set(df["AwayTeam"]))
team_id_by_name = {name: det_id("team", f"{PROVIDER}|{name}") for name in team_names}

elo = {name: INITIAL_ELO for name in team_names}
pre_match_elos = {}

match_rows = []; pred_rows = []; bench_rows = []; stats_rows = []

# Goals + red cards per match from the ESPN events notebook. Every match
# ESPN covered gets an entry ([] for a 0-0 with no reds); matches missing
# from the dict have no per-event data and use the HT/FT approximation.
def load_timeline_events():
    out = {}
    conn = connect()
    try:
        with conn.cursor() as cur:
            cur.execute("SELECT DISTINCT match_id FROM match_events WHERE provider = 'espn'")
            for (mid,) in cur.fetchall(): out[mid] = []
            cur.execute("""SELECT match_id, event_type, team_side, minute, COALESCE(added_minute, 0)
                           FROM match_events
                           WHERE provider = 'espn' AND event_type IN ('GOAL', 'OWN_GOAL', 'RED_CARD')
                             AND team_side IN ('HOME', 'AWAY') AND minute IS NOT NULL
                           ORDER BY match_id, minute, added_minute""")
            # Own goals are already credited to the beneficiary side.
            for mid, etype, side, minute, added in cur.fetchall():
                kind = "RED_CARD" if etype == "RED_CARD" else "GOAL"
                out[mid].append((int(minute), int(added), kind, side))
    except pymysql.err.ProgrammingError:
        pass  # match_events table not created yet
    finally:
        conn.close()
    return out

events_by_match = load_timeline_events()

def _state_at(events, minute):
    """(home goals, away goals, home reds, away reds) up to and including
    `minute`; a 45'+3 goal counts towards the half-time state."""
    hs = as_ = hr = ar = 0
    for m, _, kind, side in events:
        if m > minute: break
        if kind == "GOAL":
            if side == "HOME": hs += 1
            else: as_ += 1
        elif side == "HOME": hr += 1
        else: ar += 1
    return hs, as_, hr, ar

def make_pred_rows(match_id, kickoff, home_elo, away_elo, hs_ht, as_ht, hs_ft, as_ft, events=None):
    out = []
    # Trust the events only when their goals add up to the CSV final score.
    use_events = (events is not None and hs_ft is not None
                  and _state_at(events, 200)[:2] == (hs_ft, as_ft))
    score_source = "events" if use_events else "ht-ft-approximation"
    def add(minute, hs, as_, origin, forecast_at, hr=0, ar=0):
        p_h, p_d, p_a = predict(home_elo, away_elo, hs, as_, minute, hr, ar)
        feats = {"home_elo": home_elo, "away_elo": away_elo, "minute": minute,
                 "home_score": hs, "away_score": as_, "home_red": hr, "away_red": ar,
                 "home_advantage": HOME_ADVANTAGE, "model": MODEL_NAME, "model_version": MODEL_VERSION,
                 "source_provider": PROVIDER, "score_source": score_source, "origin": origin}
        pred_id = det_id("prediction", f"{match_id}|{MODEL_NAME}|{MODEL_VERSION}|{minute}|{origin}")
        out.append((pred_id, match_id, MODEL_NAME, MODEL_VERSION, forecast_at,
                    None if minute == 0 else minute, hs, as_, hr, ar,
                    p_h, p_d, p_a, json.dumps(feats), "HISTORICAL", forecast_at))
    add(0, 0, 0, "pre-match", kickoff - timedelta(hours=1))
    if hs_ft is None or as_ft is None: return out
    if use_events:
        grid = set(range(5, 90, TIMELINE_STEP_MIN)) | {45, 90}
        for minute in range(5, 90, TIMELINE_STEP_MIN):
            if minute == 45: continue
            hs, as_, hr, ar = _state_at(events, minute)
            add(minute, hs, as_, "reconstructed-5min", kickoff + timedelta(minutes=minute), hr, ar)
        seen = set()
        for m, added, kind, _ in events:
            # Grid snapshots at the same minute already include the event.
            if m in grid or m in seen or m <= 0: continue
            seen.add(m)
            hs, as_, hr, ar = _state_at(events, m)
            add(m, hs, as_, "goal-event" if kind == "GOAL" else "red-card-event",
                kickoff + timedelta(minutes=m + added), hr, ar)
        hs, as_, hr, ar = _state_at(events, 45)
        add(45, hs, as_, "half-time", kickoff + timedelta(minutes=45), hr, ar)
        _, _, hr, ar = _state_at(events, 200)
        add(90, hs_ft, as_ft, "full-time", kickoff + timedelta(minutes=90), hr, ar)
        return out
    for minute in range(5, 90, TIMELINE_STEP_MIN):
        if minute == 45: continue
        hs, as_ = (0, 0) if minute < 45 else (hs_ht, as_ht)
        add(minute, hs, as_, "reconstructed-5min", kickoff + timedelta(minutes=minute))
    add(45, hs_ht, as_ht, "half-time", kickoff + timedelta(minutes=45))
    add(90, hs_ft, as_ft, "full-time", kickoff + timedelta(minutes=90))
    return out

def make_bench_row(row, match_id, kickoff, hs_ft, as_ft):
    oh, od, oa = row.get("AvgH"), row.get("AvgD"), row.get("AvgA"); source = "market-avg"
    if pd.isna(oh) or pd.isna(od) or pd.isna(oa):
        oh, od, oa = row.get("B365H"), row.get("B365D"), row.get("B365A"); source = "bet365"
    if pd.isna(oh) or pd.isna(od) or pd.isna(oa): return None
    try: oh, od, oa = float(oh), float(od), float(oa)
    except (TypeError, ValueError): return None
    if min(oh, od, oa) <= 1.0: return None
    rh, rd, ra = 1/oh, 1/od, 1/oa; overround = rh + rd + ra
    ph, pd_, pa = rh/overround, rd/overround, ra/overround
    bench_id = det_id("benchmark", f"{match_id}|football-data-couk-{source}|pre-match")
    raw = json.dumps({"source": source, "AvgH": None if pd.isna(row.get("AvgH")) else float(row.get("AvgH")),
                      "AvgD": None if pd.isna(row.get("AvgD")) else float(row.get("AvgD")),
                      "AvgA": None if pd.isna(row.get("AvgA")) else float(row.get("AvgA")),
                      "B365H": None if pd.isna(row.get("B365H")) else float(row.get("B365H")),
                      "B365D": None if pd.isna(row.get("B365D")) else float(row.get("B365D")),
                      "B365A": None if pd.isna(row.get("B365A")) else float(row.get("B365A"))})
    forecast_at = kickoff - timedelta(hours=1)
    return (bench_id, match_id, f"football-data-couk-{source}", "v1", "MARKET_ODDS", "PRE_MATCH",
            forecast_at, forecast_at, None,
            "FINISHED" if hs_ft is not None else "SCHEDULED",
            hs_ft or 0, as_ft or 0, 0, 0, ph, pd_, pa, raw, overround, "HISTORICAL")

def _int_or_none(v):
    if v is None or pd.isna(v): return None
    try: return int(v)
    except (TypeError, ValueError): return None

def _float_or_none(v):
    if v is None or pd.isna(v): return None
    try: return float(v)
    except (TypeError, ValueError): return None

def _str_or_none(v):
    if v is None or pd.isna(v): return None
    s = str(v).strip()
    return s or None

def make_stats_row(row, match_id):
    stats = {
        "home_shots": _int_or_none(row.get("HS")), "away_shots": _int_or_none(row.get("AS")),
        "home_shots_on_t": _int_or_none(row.get("HST")), "away_shots_on_t": _int_or_none(row.get("AST")),
        "home_corners": _int_or_none(row.get("HC")), "away_corners": _int_or_none(row.get("AC")),
        "home_fouls": _int_or_none(row.get("HF")), "away_fouls": _int_or_none(row.get("AF")),
        "home_yellow": _int_or_none(row.get("HY")), "away_yellow": _int_or_none(row.get("AY")),
        "home_red": _int_or_none(row.get("HR")), "away_red": _int_or_none(row.get("AR")),
        "home_xg": _float_or_none(row.get("HxG")), "away_xg": _float_or_none(row.get("AxG")),
        "referee": _str_or_none(row.get("Referee")),
    }
    if all(v is None for v in stats.values()): return None
    return (match_id, stats["home_shots"], stats["away_shots"],
            stats["home_shots_on_t"], stats["away_shots_on_t"],
            stats["home_corners"], stats["away_corners"],
            stats["home_fouls"], stats["away_fouls"],
            stats["home_yellow"], stats["away_yellow"],
            stats["home_red"], stats["away_red"],
            stats["home_xg"], stats["away_xg"],
            stats["referee"], PROVIDER)

for idx, row in df.iterrows():
    home, away = row["HomeTeam"], row["AwayTeam"]
    kickoff = row["KickoffAt"].to_pydatetime()
    pre_match_elos[idx] = (elo[home], elo[away])
    played = not (pd.isna(row.get("FTHG")) or pd.isna(row.get("FTAG")))
    ext = f"{COMPETITION_SEASON}|{kickoff:%Y-%m-%d}|{home}|{away}"
    match_id = det_id("match", f"{PROVIDER}|{ext}")
    if played:
        hs_ft, as_ft = int(row["FTHG"]), int(row["FTAG"])
        hs_ht = int(row["HTHG"]) if not pd.isna(row.get("HTHG")) else 0
        as_ht = int(row["HTAG"]) if not pd.isna(row.get("HTAG")) else 0
        fr = "HOME" if hs_ft > as_ft else "AWAY" if hs_ft < as_ft else "DRAW"
        hr = int(row["HR"]) if not pd.isna(row.get("HR")) else 0
        ar = int(row["AR"]) if not pd.isna(row.get("AR")) else 0
        status, minute, hs_db, as_db = "FINISHED", 90, hs_ft, as_ft
    else:
        hs_ft = as_ft = hs_ht = as_ht = hr = ar = 0; fr = None
        status, minute, hs_db, as_db = "SCHEDULED", None, 0, 0
    match_rows.append((match_id, PROVIDER, ext, comp_id,
                       team_id_by_name[home], team_id_by_name[away], kickoff, status, minute,
                       hs_db, as_db, hr, ar, fr, None, "HISTORICAL", now_utc))
    he, ae = pre_match_elos[idx]
    if played:
        pred_rows.extend(make_pred_rows(match_id, kickoff, he, ae, hs_ht, as_ht, hs_ft, as_ft,
                                        events_by_match.get(match_id)))
        b = make_bench_row(row, match_id, kickoff, hs_ft, as_ft)
        if b: bench_rows.append(b)
        st = make_stats_row(row, match_id)
        if st: stats_rows.append(st)
        elo[home], elo[away] = _elo_update(elo[home], elo[away], hs_ft, as_ft)
    else:
        pred_rows.extend(make_pred_rows(match_id, kickoff, he, ae, 0, 0, None, None))
        b = make_bench_row(row, match_id, kickoff, None, None)
        if b: bench_rows.append(b)

team_rows = [(team_id_by_name[n], PROVIDER, n, n, n[:50], COMPETITION_COUNTRY, float(elo[n])) for n in team_names]
comp_row = (comp_id, PROVIDER, COMPETITION_EXT_ID, COMPETITION_NAME, COMPETITION_COUNTRY, COMPETITION_SEASON)
event_driven = sorted({r[1] for r in pred_rows if json.loads(r[13])["score_source"] == "events"})
played = sum(1 for r in match_rows if r[7] == "FINISHED")
print(f"teams={len(team_rows)}  matches={len(match_rows)}  preds={len(pred_rows)}  bench={len(bench_rows)}  "
      f"stats={len(stats_rows)}  event-driven timelines={len(event_driven)}/{played}")


## 6 – Upsert all rows (idempotent via `ON DUPLICATE KEY UPDATE`)

In [ ]:
def _chunked(seq, size):
    for i in range(0, len(seq), size): yield seq[i:i+size]

def upsert_many(cur, table, cols, update_cols, rows, chunk_size=500):
    if not rows: return 0
    placeholders = "(" + ",".join(["%s"] * len(cols)) + ")"
    upd = ", ".join(f"{c}=VALUES({c})" for c in update_cols)
    total = 0
    for chunk in _chunked(rows, chunk_size):
        vals = ",".join([placeholders] * len(chunk))
        q = f"INSERT INTO {table} ({', '.join(cols)}) VALUES {vals} ON DUPLICATE KEY UPDATE {upd}"
        flat = []
        for r in chunk: flat.extend(r)
        cur.execute(q, flat); total += len(chunk)
    return total

COMP_COLS=["id","provider","external_id","name","country","season"]; COMP_UPD=["name","country","season"]
TEAM_COLS=["id","provider","external_id","name","short_name","country","elo_rating"]; TEAM_UPD=["name","short_name","country","elo_rating"]
MATCH_COLS=["id","provider","external_id","competition_id","home_team_id","away_team_id","kickoff_at","status","minute",
            "home_score","away_score","home_red_cards","away_red_cards","final_result","venue","provenance","source_updated_at"]
MATCH_UPD=["status","minute","home_score","away_score","home_red_cards","away_red_cards","final_result","venue","provenance","source_updated_at"]
PRED_COLS=["id","match_id","model_name","model_version","forecast_at","match_minute","home_score_at_fc","away_score_at_fc",
           "home_red_cards_fc","away_red_cards_fc","p_home","p_draw","p_away","features_json","provenance","source_timestamp"]
PRED_UPD=["forecast_at","match_minute","home_score_at_fc","away_score_at_fc","home_red_cards_fc","away_red_cards_fc",
          "p_home","p_draw","p_away","features_json","provenance","source_timestamp"]
BENCH_COLS=["id","match_id","provider","provider_version","source_type","horizon","forecast_at","source_timestamp",
            "match_minute","match_status","home_score_at_fc","away_score_at_fc","home_red_cards_fc","away_red_cards_fc",
            "p_home","p_draw","p_away","raw_odds_json","overround","provenance"]
BENCH_UPD=["forecast_at","source_timestamp","match_status","home_score_at_fc","away_score_at_fc",
           "p_home","p_draw","p_away","raw_odds_json","overround","provenance"]
STATS_COLS=["match_id","home_shots","away_shots","home_shots_on_t","away_shots_on_t",
            "home_corners","away_corners","home_fouls","away_fouls",
            "home_yellow","away_yellow","home_red","away_red",
            "home_xg","away_xg","referee","source"]
STATS_UPD=["home_shots","away_shots","home_shots_on_t","away_shots_on_t",
           "home_corners","away_corners","home_fouls","away_fouls",
           "home_yellow","away_yellow","home_red","away_red",
           "home_xg","away_xg","referee","source"]

t0 = time.time()
conn = connect()
try:
    with conn.cursor() as cur:
        # Event-minute snapshots move if ESPN corrects a minute, so drop
        # them before re-inserting the current set.
        for chunk in _chunked(event_driven, 500):
            cur.execute(f"""DELETE FROM model_predictions
                            WHERE model_name = %s AND model_version = %s
                              AND match_id IN ({",".join(["%s"] * len(chunk))})
                              AND JSON_EXTRACT_STRING(features_json, 'origin') IN ('goal-event', 'red-card-event')""",
                        (MODEL_NAME, MODEL_VERSION, *chunk))
        upsert_many(cur, "competitions", COMP_COLS, COMP_UPD, [comp_row])
        upsert_many(cur, "teams", TEAM_COLS, TEAM_UPD, team_rows)
        upsert_many(cur, "matches", MATCH_COLS, MATCH_UPD, match_rows)
        upsert_many(cur, "model_predictions", PRED_COLS, PRED_UPD, pred_rows)
        upsert_many(cur, "benchmark_predictions", BENCH_COLS, BENCH_UPD, bench_rows)
        upsert_many(cur, "match_stats", STATS_COLS, STATS_UPD, stats_rows)

        run_id = det_id("ingest-run", f"{PROVIDER}|{datetime.now(timezone.utc):%Y-%m-%dT%H}")
        details = json.dumps({"csv_url": CSV_URL, "season": COMPETITION_SEASON,
                              "played_matches": sum(1 for r in match_rows if r[7] == "FINISHED"),
                              "model": MODEL_NAME, "model_version": MODEL_VERSION})
        cur.execute("""INSERT INTO data_ingestion_runs
                       (id, provider, job_name, status, started_at, finished_at,
                        records_in, records_out, error_message, details_json)
                       VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
                       ON DUPLICATE KEY UPDATE status=VALUES(status),
                         finished_at=VALUES(finished_at), records_in=VALUES(records_in),
                         records_out=VALUES(records_out), error_message=VALUES(error_message),
                         details_json=VALUES(details_json)""",
                    (run_id, PROVIDER, "notebook-pl-season", "SUCCEEDED",
                     datetime.fromtimestamp(t0, tz=timezone.utc).replace(tzinfo=None),
                     datetime.now(timezone.utc).replace(tzinfo=None),
                     len(df), len(match_rows) + len(pred_rows) + len(bench_rows) + len(stats_rows),
                     None, details))
finally:
    conn.close()
print(f"done in {time.time() - t0:.1f}s")


## 7 – Sanity check (optional)

Run this cell after an ingest to confirm the row counts and the top-10 Elo leaderboard.

In [ ]:
conn = connect()
try:
    with conn.cursor() as cur:
        for t in ("competitions","teams","matches","model_predictions","benchmark_predictions","match_stats","data_ingestion_runs"):
            cur.execute(f"SELECT COUNT(*) FROM {t}")
            print(f"  {t:<24} {cur.fetchone()[0]}")
        print()
        cur.execute("SELECT name, elo_rating FROM teams ORDER BY elo_rating DESC LIMIT 10")
        print("Top 10 teams by Elo:")
        for n, e in cur.fetchall():
            print(f"  {n:<18} {e:.1f}")
finally:
    conn.close()
